In [2]:
from fairchem.core.datasets import AseDBDataset
from tqdm import tqdm
import numpy as np
import torch
from torch.utils.data import Subset
from IPython.display import Image, display
import ase

/pscratch/sd/y/yuejian/envs/fairchemV2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
# load dataset
dataset_path = "/global/homes/y/yuejian/project/MLFF-distill/yuejian/electrolytes/toy/0000.aselmdb"
a2g_args = {  
    "molecule_cell_size": 120.0,
    "r_energy": True,
    "r_forces": True,
    # "r_stress": True,
    "r_data_keys": [ 'spin','charge', "data_id"],
    # 'sid': 'data_id',

}
# select_args ={ "data_id":"ani2x"}
# select_args ={"selection": [("charge", "=", -1)] }
select_args = {
    "filter": lambda row: row._data.get("data_id") == "orbnet_denali"
}
dataset = AseDBDataset({
                        "src": dataset_path,
                        "a2g_args": a2g_args,
                        # "select_args": select_args,
                        })

In [4]:
# load dataset
dataset_path = "/home/yuejian/project/MLFF-distill/OMOL/TOY/ligand_pocket_300/train/data0000.aselmdb"
a2g_args = {  
    "molecule_cell_size": 120.0,
    "r_energy": True,
    "r_forces": True,
    # "r_stress": True,
    "r_data_keys": [ 'spin','charge', "data_id"],
    # 'sid': 'data_id',

}
# select_args ={ "data_id":"ani2x"}
# select_args ={"selection": [("charge", "=", -1)] }
select_args = {
    "filter": lambda row: row._data.get("data_id") == "orbnet_denali"
}
dataset_2 = AseDBDataset({
                        "src": dataset_path,
                        "a2g_args": a2g_args,
                        # "select_args": select_args,
                        })

In [14]:
len(dataset)

10

In [8]:
dataset[2],dataset_2[2]

(AtomicData(atomic_numbers=[273], batch=[273], cell=[1, 3, 3], cell_offsets=[0, 3], charge=[1], data_id=[1], edge_index=[2, 0], energy=[1], fixed=[273], forces=[273, 3], natoms=[1], nedges=[1], pbc=[1, 3], pos=[273, 3], sid=[1], source=[1], spin=[1], tags=[273]),
 AtomicData(atomic_numbers=[90], batch=[90], cell=[1, 3, 3], cell_offsets=[0, 3], charge=[1], data_id=[1], edge_index=[2, 0], energy=[1], fixed=[90], forces=[90, 3], natoms=[1], nedges=[1], pbc=[1, 3], pos=[90, 3], sid=[1], source=[1], spin=[1], tags=[90]))

In [11]:
for i in tqdm(range(len(dataset))):
    data = dataset[i]
    # print(data)
    # print(data['data_id'])
    # print(data['energy'].shape, data['forces'].shape, data['stress'].shape)
    # print(data['cell'].shape, data['positions'].shape, data['atomic_numbers'].shape)
    # print(data['spin'], data['charge'])
    # print(data['data_id'])
    # display(Image(ase.visualize.view(data)))
    # print("====================================")
    if data.natoms[0]<40:
        print(i)
        break

100%|██████████| 805/805 [00:01<00:00, 426.42it/s]


In [3]:
len(dataset)

92808

### read in aselmdb

In [8]:
# Source and destination paths
src_path = "/global/homes/y/yuejian/project/MLFF-distill/yuejian/electrolytes/aug_29_first_100ps/napf6/train/data.0000.aselmdb"
dst_path = "/global/homes/y/yuejian/project/MLFF-distill/yuejian/electrolytes/toy/0000.aselmdb"

connect_args = {
    "readonly": False,
    "use_lock_file": False,
}

# Open source DB in readonly mode
src_db = ase.db.connect(src_path, **connect_args)
# Open/create destination DB for writing
dst_db = ase.db.connect(dst_path, use_lock_file=False)

In [9]:
len(src_db)  # Get the number of rows in the source DB

11438

In [5]:
# for i in range(210):
i = 0
for row in tqdm(src_db.select(),total=len(src_db)):
    i+=1
    if i > 1000:
        break
    if i > 990:
        atoms = row.toatoms()  # Get Atoms object
        data = dict(row.data) if hasattr(row, 'data') else {}
        key = getattr(row, 'key', None)
        print(atoms,data)
        # print(f"Processing row with key: {key}")
        # Write to the new database, preserving key and metadata
        # dst_db.write(atoms, data=data)
        # dst_db.write(atoms, key=key, data=data)
    break

  0%|          | 0/11438 [00:00<?, ?it/s]


In [10]:
# for i in range(210):
i = 0
for row in tqdm(src_db.select(),total=len(src_db)):
    i+=1
    if i > 10:
        break
    atoms = row.toatoms()  # Get Atoms object
    data = dict(row.data) if hasattr(row, 'data') else {}
    key = getattr(row, 'key', None)
    # print(f"Processing row with key: {key}")
    # Write to the new database, preserving key and metadata
    dst_db.write(atoms, data=data)
    # dst_db.write(atoms, key=key, data=data)

  0%|          | 10/11438 [00:00<12:04, 15.78it/s]


In [29]:
# for i in range(210):
i = 0
for row in tqdm(src_db.select(),total=len(src_db)):
    i+=1
    if i > 10000:
        break
    atoms = row.toatoms()  # Get Atoms object
    data = dict(row.data) if hasattr(row, 'data') else {}
    key = getattr(row, 'key', None)
    # print(f"Processing row with key: {key}")
    # Write to the new database, preserving key and metadata
    dst_db.write(atoms, data=data)
    # dst_db.write(atoms, key=key, data=data)


 50%|████▉     | 10000/20137 [00:28<00:29, 346.44it/s]


In [11]:
src_db.close()  # Close the source database
dst_db.close()  # Close the destination database

In [6]:
# load dataset
dataset_path = "/home/yuejian/project/MLFF-distill/test_dump_aselmdb/source"
a2g_args = {  
    "molecule_cell_size": 120.0,
    "r_energy": True,
    "r_forces": True,
    # "r_stress": True,
    "r_data_keys": [ 'spin','charge', "data_id"],
    # 'sid': 'data_id',

}
# select_args ={ "data_id":"ani2x"}
# select_args ={"selection": [("charge", "=", -1)] }
select_args = {
    "filter": lambda row: row._data.get("data_id") == "orbnet_denali"
}
source_dataset = AseDBDataset({
                        "src": dataset_path,
                        "a2g_args": a2g_args,
                        # "select_args": select_args,
                        })

In [7]:
# load dataset
dataset_path = "/home/yuejian/project/MLFF-distill/test_dump_aselmdb/copy"
a2g_args = {  
    "molecule_cell_size": 120.0,
    "r_energy": True,
    "r_forces": True,
    # "r_stress": True,
    "r_data_keys": [ 'spin','charge', "data_id"],
    # 'sid': 'data_id',

}
# select_args ={ "data_id":"ani2x"}
# select_args ={"selection": [("charge", "=", -1)] }
select_args = {
    "filter": lambda row: row._data.get("data_id") == "orbnet_denali"
}
copy_dataset = AseDBDataset({
                        "src": dataset_path,
                        "a2g_args": a2g_args,
                        # "select_args": select_args,
                        })

In [27]:
for source,copy in tqdm(zip(source_dataset, copy_dataset), total=len(source_dataset)):
    # check if each argument matches between source and copy
    # AtomicData(atomic_numbers=[14], batch=[14], cell=[1, 3, 3], cell_offsets=[0, 3], charge=[1], data_id=[1], edge_index=[2, 0], energy=[1], fixed=[14], forces=[14, 3], natoms=[1], nedges=[1], pbc=[1, 3], pos=[14, 3], sid=[1], source=[1], spin=[1], tags=[14])
    for key in source.keys():
        if key not in copy.keys():
            print(f"Key {key} not found in copy dataset")
            continue
        if isinstance(source[key], torch.Tensor):
            if not torch.equal(source[key], copy[key]):
                print(f"Mismatch in tensor for key {key}")
        elif isinstance(source[key], np.ndarray):
            if not np.array_equal(source[key], copy[key]):
                print(f"Mismatch in array for key {key}")
        else:
            if source[key] != copy[key]:
                print(f"Mismatch in value for key {key}: {source[key]} != {copy[key]}")
    

100%|██████████| 49835/49835 [01:10<00:00, 707.43it/s]


In [20]:
copy.keys()

{'atomic_numbers',
 'batch',
 'cell',
 'cell_offsets',
 'charge',
 'data_id',
 'edge_index',
 'energy',
 'fixed',
 'forces',
 'natoms',
 'nedges',
 'pbc',
 'pos',
 'sid',
 'source',
 'spin',
 'tags'}

In [14]:
source

AtomicData(atomic_numbers=[14], batch=[14], cell=[1, 3, 3], cell_offsets=[0, 3], charge=[1], data_id=[1], edge_index=[2, 0], energy=[1], fixed=[14], forces=[14, 3], natoms=[1], nedges=[1], pbc=[1, 3], pos=[14, 3], sid=[1], source=[1], spin=[1], tags=[14])

In [11]:
source.source

['ani1xbb/aniBB_014_569966_0_1/orca.tar.zst']

In [26]:
source_dataset[5].pos[:10,:], copy_dataset[5].pos[:10,:]

(tensor([[128.2628, 138.1711, 127.4267],
         [127.7452, 137.1824, 126.3279],
         [127.7188, 137.6234, 125.1517],
         [127.5193, 138.2895, 128.2149],
         [129.2468, 137.9889, 127.8587],
         [128.3287, 139.1104, 126.8776],
         [127.5184, 135.9038, 126.6964],
         [126.8101, 134.9888, 125.8729],
         [125.5702, 134.3704, 126.4981],
         [125.6232, 133.7381, 127.5706]]),
 tensor([[128.2628, 138.1711, 127.4267],
         [127.7452, 137.1824, 126.3279],
         [127.7188, 137.6234, 125.1517],
         [127.5193, 138.2895, 128.2149],
         [129.2468, 137.9889, 127.8587],
         [128.3287, 139.1104, 126.8776],
         [127.5184, 135.9038, 126.6964],
         [126.8101, 134.9888, 125.8729],
         [125.5702, 134.3704, 126.4981],
         [125.6232, 133.7381, 127.5706]]))